In [ ]:
!pip install datasets tiktoken -q

In [ ]:
import os, numpy as np, tiktoken
from datasets import load_dataset

In [ ]:
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

TRAIN_BIN     = "/kaggle/working/train.bin"
VAL_BIN       = "/kaggle/working/val.bin"

In [ ]:
if os.path.exists(TRAIN_BIN) and os.path.exists(VAL_BIN) \
   and os.path.getsize(TRAIN_BIN) > 0 and os.path.getsize(VAL_BIN) > 0:
    print("Binary files exist, skipping tokenization.")
else:
    
    enc = tiktoken.get_encoding("gpt2")
    train_tokens, val_tokens = [], []

    print("Loading TinyStories (train + validation)...")

    train_dataset = load_dataset("roneneldan/TinyStories", split="train")
    val_dataset   = load_dataset("roneneldan/TinyStories", split="validation")

    print("Tokenizing train split...")
    for i, example in enumerate(train_dataset):
        train_tokens.extend(enc.encode(example["text"] + "\n"))

        if i % 1000 == 0:
            print(f"  train: {len(train_tokens)/1e6:.1f}M tokens")

    print("Tokenizing validation split...")
    for i, example in enumerate(val_dataset):
        val_tokens.extend(enc.encode(example["text"] + "\n"))

        if i % 1000 == 0:
            print(f"  val: {len(val_tokens)/1e6:.1f}M tokens")

    # save
    np.array(train_tokens, dtype=np.uint16).tofile(TRAIN_BIN)
    np.array(val_tokens,   dtype=np.uint16).tofile(VAL_BIN)

    print("saved.")

In [ ]:
train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")
print(f"train: {len(train_data):,} | val: {len(val_data):,}")
assert train_data.max() < 50257, f"OOB token in train: {train_data.max()}"
assert val_data.max()   < 50257, f"OOB token in val: {val_data.max()}"
print("tokens OK")